# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    last_err = None
    for attempt in range(tries):
        try:
            r = requests.get(BASE, params=params, timeout=30)
        except (requests.ConnectionError, requests.Timeout) as e:
            last_err = e
        else:
            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])
            err = requests.HTTPError(
                f'{r.status_code} from Open-Meteo: {r.text[:200]}', response=r
            )
            if r.status_code not in RETRY_STATUSES:
                raise err
            last_err = err
        if attempt < tries - 1:
            time.sleep(2 ** attempt)  # 1s, 2s, 4s, ...
    raise RuntimeError(f'get_weather failed after {tries} attempts') from last_err

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
dates = ['2026-09-05', '2026-10-03']
used_fallback = False

try:
    df = get_weather(min(dates), max(dates))   # one request, 29 days
    pulled_at = pd.Timestamp.now(tz='America/New_York')
    wx = df[df['time'].isin(dates)].reset_index(drop=True)
except (requests.RequestException, RuntimeError) as e:
    print('request failed')
    wx = None

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

if wx is None or len(wx) != len(dates) or wx.isna().any().any():
    used_fallback = True
    wx = FALLBACK.copy()

wx['source'] = 'fallback sample' if used_fallback else 'open-meteo'
wx

,time,temperature_2m_max,precipitation_sum,source
0,2026-09-05,29.4,2.5,open-meteo
1,2026-10-03,22.0,14.7,open-meteo


### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [4]:
wx = (
    wx.rename(columns={
        'time': 'date',
        'temperature_2m_max': 'temp_max',
        'precipitation_sum': 'precip_mm',
    })
    .assign(date=lambda d: pd.to_datetime(d['date'], format='%Y-%m-%d'))
    [['date', 'temp_max', 'precip_mm', 'source']]
)
wx

,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
1,2026-10-03,22.0,14.7,open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [5]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [6]:
print("2026-10-03 was wetter as it had a precipitation measurement of 14.7 mm compared to 2.5 mm, which is a difference of 12.2 mm")

2026-10-03 was wetter as it had a precipitation measurement of 14.7 mm compared to 2.5 mm, which is a difference of 12.2 mm


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [7]:
import inspect

sig = inspect.signature(get_weather).parameters
lat, lon = sig['lat'].default, sig['lon'].default

print(f'Endpoint:    {BASE}')
print(f'Coordinates: {lat}, {lon}')
print(f'Date range:  {min(dates)} to {max(dates)} (kept {", ".join(dates)})')
if used_fallback:
    print('Pulled:      no usable pull; the live request failed or came back incomplete')
    print('Data:        FALLBACK SAMPLE, hard-coded placeholder values, not observations')
else:
    print(f'Pulled:      {pulled_at:%Y-%m-%d %H:%M %Z}')
    print('Data:        real, Open-Meteo historical archive (reanalysis model, not a station gauge)')

Endpoint:    https://archive-api.open-meteo.com/v1/archive
Coordinates: 38.03, -78.51
Date range:  2026-09-05 to 2026-10-03 (kept 2026-09-05, 2026-10-03)
Pulled:      2026-10-10 16:32 EDT
Data:        real, Open-Meteo historical archive (reanalysis model, not a station gauge)


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

The merge woukd still work. A left merge keeps all 200 sales rows and, because there is no weather to match, fills the rain column with NaN for every day. Then the chart then has nothing to plot and there'd be get empty axes, or if anyone filled the blanks with zero along the way, a flat line meaning it never rained. Either way it reads as rain has no effect on revenue.

Nobody catches it because nothing fails, there is no error, the table still has 200 rows, and the chart renders. “No relationship” is a believable result, so it looks like a finding instead of a bug. The real problem happened several steps earlier and left no trace. That is why the function should raise, as a loud failure at the source is much better and usuable than a quiet wrong answer at the end.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [8]:
# stretch